In [2]:
#LIBRARIES

import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.ensemble import IsolationForest
import os
import glob
import re
import yaml

In [7]:
# GLOBAL VARIABLES

#directories and paths
DATASET_PATH = "/home/marina/Desktop/ai_algorithms/ws25_aia_complete_data"
LOGS_DIR = "logs"
SCENARIO_SEP = "maps-"
LOG_CCI = "component_container_isolated"


#NAMES COLUMNS
TIMESTAMP = 'timestamp'
FRAME = 'frame'
POSITION_X = "position.x"
POSITION_Y = "position.y"
ORIENTATION_YAW = 'orientation.yaw'

#FRAMES
REAL_POSE = 'nav2_turtlebot4_base_link_gt'
ESTIMATED_POSE = 'base_link'

#PATERN_LOG
PATTERN_BR = r"\[(.*?)\]"

#LOCALIZATION ERROR
COUNT_IN_CONSISTENT = 3

In [8]:
# FUNCTIONS

def compute_path_length(x: np.ndarray, y: np.ndarray) -> float:
    dx = np.diff(x)
    dy = np.diff(y)
    return float(np.sum(np.sqrt(dx ** 2 + dy ** 2)))

In [9]:
#Objects

# Class with the data of a ros iteration in a scenario
class DataProcessed:
    def __init__(self, scenario, run_n, csv_file):
        #general info
        self.scenario = scenario
        self.run_n = run_n
        
        #csv info
        self.csv_file = csv_file
        if os.path.exists(csv_file):
            self.df = pd.read_csv(
                csv_file,
                comment='#',       
                sep=',',           
                quotechar='"',     
                engine='python'
            )
        else:
            #csv does not exist
            self.csv_file = None
            
        self.position_error = np.array([])
        self.yaw_error = np.array([])
        
        #logs info
        self.logs = []
        self.failure_run = True
        self.init_t = None
        self.end_t = None

        #scenario info
        self.scenario_info = None
        
        #anomalies info
        self.anomalies = []
        self.unique_anomalies = []

        #IsolationForest navigation score 
        metrics = None
        iso_score = None
        

    def add_log(self, log):
        self.logs.append(log)

        #special logs information about begining and end of a run
        if 'Begin navigating' in log.msg:
            self.init_t = log.time
        if 'Goal succeeded' in log.msg:
            self.failure_run = False
            self.end_t = log.time
        if 'Goal failed' in log.msg:
            self.end_t = log.time

    def load_scenario_config(self, path):
        """
        Load a scenario YAML file and extract structured information.

        Parameters:
            path (str): Path to the YAML config file.

        Returns:
            dict: A dictionary containing the parsed scenario information.
        """
        with open(path, 'r') as f:
            data = yaml.safe_load(f)

        # The root key is 'test_scenario'
        scenario = data.get("test_scenario", {})

        # ---- Extract goal poses ----
        goal_poses_raw = scenario.get("goal_poses", [])
        goal_poses = []
        for g in goal_poses_raw:
            goal_poses.append({
                "x": float(g["position"].get("x", 0.0)),
                "y": float(g["position"].get("y", 0.0)),
                "yaw": float(g["orientation"].get("yaw", 0.0))
            })

        # ---- Extract start pose ----
        start_raw = scenario.get("start_pose", {})
        start_pose = {
            "x": float(start_raw.get("position", {}).get("x", 0.0)),
            "y": float(start_raw.get("position", {}).get("y", 0.0)),
            "yaw": float(start_raw.get("orientation", {}).get("yaw", 0.0))
        }

        # ---- Extract static objects ----
        objects_raw = scenario.get("static_objects", [])
        static_objects = []
        for obj in objects_raw:
            spawn_pose = obj.get("spawn_pose", {})
            position = spawn_pose.get("position", {})
            orientation = spawn_pose.get("orientation", {})
            static_objects.append({
                "name": obj.get("entity_name", ""),
                "model": obj.get("model", ""),
                "x": float(position.get("x", 0.0)),
                "y": float(position.get("y", 0.0)),
                "z": float(position.get("z", 0.0)),
                "yaw": float(orientation.get("yaw", 0.0)),
                "xacro_arguments": obj.get("xacro_arguments", "")
            })

        # ---- Build output structure ----
        scenario_info = {
            "goal_poses": goal_poses,
            "start_pose": start_pose,
            "map_file": scenario.get("map_file", ""),
            "mesh_file": scenario.get("mesh_file", ""),
            "laser_noise_std": float(scenario.get("laserscan_gaussian_noise_std_deviation", 0)),
            "laser_drop_pct": float(scenario.get("laserscan_random_drop_percentage", 0)),
            "static_objects": static_objects
        }

        # Save to the instance
        self.scenario_info = scenario_info
        return scenario_info
    
    # Get Pose error of the scenario (2D) in form of position_error, yaw_error, timestamp
    # given deduced pose info and real pose info
    def calculate_pose_error(self):
        if self.csv_file == None:
            return
        
        deduced_pose = self.df.loc[self.df[FRAME] == ESTIMATED_POSE]
        real_pose = self.df.loc[self.df[FRAME] == REAL_POSE]
        
        #since is 2D just take position.x and y and orientation.z and w.
        deduced_position_x = deduced_pose[POSITION_X].to_numpy()
        deduced_position_y = deduced_pose[POSITION_Y].to_numpy()
        deduced_yaw = deduced_pose[ORIENTATION_YAW].to_numpy()
        deduced_timestamp = deduced_pose[TIMESTAMP].to_numpy()
    
        real_position_x = real_pose[POSITION_X].to_numpy()
        real_position_y = real_pose[POSITION_Y].to_numpy()
        real_yaw = real_pose[ORIENTATION_YAW].to_numpy()
        real_timestamp = real_pose[TIMESTAMP].to_numpy()
        
        #get interpolation of real_pose to timestamp of deduced_pose.
        real_position_x_inter = np.interp(deduced_timestamp, real_timestamp, real_position_x)
        real_position_y_inter = np.interp(deduced_timestamp, real_timestamp, real_position_y)
        real_yaw_inter = np.interp(deduced_timestamp, real_timestamp, real_yaw)
    
        #now it is possible to get the vectoriced error
        dx = deduced_position_x - real_position_x_inter
        dy = deduced_position_y - real_position_y_inter
        position_error = np.sqrt(dx**2 + dy**2)
        self.position_error = np.stack((np.array(position_error), np.array(deduced_timestamp)), axis=1)

        #get yaw error.
        yaw_error = deduced_yaw - real_yaw_inter
        #normalice between [-pi, pi]
        yaw_error = (yaw_error + np.pi) % (2 * np.pi) - np.pi
        self.yaw_error = np.stack((yaw_error, deduced_timestamp), axis=1)

    def get_metrics(self):
        if self.csv_file == None or self.end_t == None or self.init_t == None:
            return
        
        #Get basic metrics:
        pos_err = self.position_error[:, 0]
        yaw_err = self.yaw_error[:, 0]
        mean_pos = float(np.nanmean(pos_err))
        rmse_pos = float(np.sqrt(np.nanmean(pos_err ** 2)))
        max_pos = float(np.nanmax(pos_err))
        mae_yaw = float(np.nanmean(np.abs(yaw_err)))

        real_pose = self.df.loc[self.df[FRAME] == REAL_POSE]
        gt_x = real_pose[POSITION_X].to_numpy()
        gt_y = real_pose[POSITION_Y].to_numpy()
        executed_len = compute_path_length(gt_x, gt_y) 

        time_spent = self.end_t - self.init_t
        
        metrics = {
            'mean_pos_error': mean_pos,
            'rmse_pos': rmse_pos,
            'max_pos_error': max_pos,
            'mae_yaw': mae_yaw,
            'executed_path_length': executed_len,
            'time_spent': time_spent
        }
        self.metrics = metrics
        return metrics


    def detect_log_anomalies(self):

        anomalies = []

        for log in self.logs:
    
            # No progress anomaly
            if "Failed to make progress" in log.msg:
                anomalies.append(("no_progress", log.time))
    
            # Planner failure
            if "Failed to create a plan" in log.msg:
                anomalies.append(("planner_failure", log.time))
    
            # Collision
            if "Collision Ahead" in log.msg:
                anomalies.append(("collision", log.time))
    
            # Stuck anomalies
            if "spin failed" in log.msg or "backup failed" in log.msg:
                anomalies.append(("stuck", log.time))
    
            # Fatal initialization
            if log.type == "FATAL":
                anomalies.append(("fatal_initialization", log.time))

            # Goal failed
            if "Goal failed" in log.msg:
                anomalies.append(("goal_failure", log.time))

        if len(self.logs) == 0:
            anomalies.append(("no_initiation", 0))

        
        self.anomalies += anomalies

    def detect_time_anomalie(self, scenario_data):
        time_run = []
        
        for data in scenario_data:
            if data.init_t is not None and data.end_t is not None:
                time_run.append(data.end_t - data.init_t)

        #since we know the number of runs is low, we use median instead of mean which also is not influenced by outliers
        if len(time_run) != 0:
            med = np.median(time_run)
            MAD = np.median(np.abs(time_run - med))  # Median of absolute desviation

            # tipic threshold: 3 * MAD
            umbral = med + 3 * MAD
            if self.init_t is not None and self.end_t is not None:
                if self.end_t - self.init_t > umbral:
                    self.anomalies.append(("time_outlier", self.end_t))
    
    def detect_localization_anomalie(self, umbral_error):
        if self.csv_file == None:
            return

        anomalies = []
        in_outlier = False
        count_in = 0

        position_sorted = self.position_error[self.position_error[:, 1].argsort()]
        # if the error is higher than the umbral error and is consisten which it means
        # there is more than one measure outside the error, it is detected the anomalie
        for p_err, t in position_sorted:
            if p_err > umbral_error:
                if not in_outlier:
                    in_outlier = True
                else:
                    if count_in == COUNT_IN_CONSISTENT:
                        anomalies.append(("position_error", t)) 
                count_in = count_in + 1
            elif p_err < umbral_error and in_outlier:
                in_outlier = False
                count_in = 0
                
        self.anomalies += anomalies
    
    def detect_navigation_performance(self, iso_tree):
        #score using the tree
        if self.csv_file == None or self.init_t is None or self.end_t is None:
            return

        summary = pd.DataFrame([self.metrics])

        num_cols = ['mean_pos_error', 'rmse_pos', 'max_pos_error', 'mae_yaw', 'executed_path_length', 'time_spent']
        feat_df = summary[num_cols].fillna(-1)
        try:   
            score = iso.decision_function(feat_df)
            self.iso_score = score[0]
            anom = iso.predict(feat_df)
            if anom[0] == -1:
                self.anomalies.append(("navigation_performance", self.end_t))
        except Exception as e:
            print('IsolationForest failed:', e)

    def clean_anomalies(self):
        self.anomalies = []

    def get_unique_anomalies(self):
        if len(self.anomalies) != 0:
            self.unique_anomalies = set(a[0] for a in self.anomalies)
        return self.unique_anomalies
        
#Class for the logs of ros
class LogsRos:
    def __init__(self, type_msg, time, sender, msg):
        self.type = type_msg
        self.time = time
        self.sender = sender
        self.msg = msg

cmap = plt.get_cmap('tab10')

In [10]:
#Get data stored and preprocessing
# ***NOW dataset is an array of objects DataProcessed. they aren´t anymore grouped by scenario.
dataset = []

for scenario_dir in os.listdir(DATASET_PATH):
    # Skip hidden files like .DS_Store
    if scenario_dir.startswith('.'):
        continue
    scenario_path = os.path.join(DATASET_PATH, scenario_dir)
    # Skip if not a directory
    if not os.path.isdir(scenario_path):
        continue
    scenario = scenario_dir.split(SCENARIO_SEP)[1]
    for run_n in os.listdir(scenario_path):
        # Skip hidden files and non-directories
        if run_n.startswith('.'):
            continue
        run_path = os.path.join(scenario_path, run_n)
        if not os.path.isdir(run_path):
            continue
        #get csv
        csv_file = os.path.join(scenario_path, run_n, "poses.csv")
        data = DataProcessed(scenario, int(run_n), csv_file)
        data.calculate_pose_error()
        #get scenario info
        scnario_info_file = os.path.join(scenario_path, run_n, "scenario.config")
        #get logs
        logs_file = glob.glob(scenario_path + "/" + run_n + "/" + LOGS_DIR + "/" + LOG_CCI + "*")
        if len(logs_file) != 0 and os.path.isfile(logs_file[0]):
            with open(logs_file[0], "r") as f:
                for line in f:
                    # get strings between []
                    brakets = re.findall(PATTERN_BR, line)
                    if len(brakets) == 3:
                        # log info are structured like [type] [time] [sender] : msg
                        type_msg = brakets[0]
                        time = float(brakets[1])
                        sender = brakets[2]
                        msg = line.split("]:")[-1].strip()
                        log = LogsRos(type_msg, time, sender, msg)
                        #Add log to the datasetRos object
                        data.add_log(log)
                        
        dataset.append(data)

In [11]:
#Get metrics

position_error_total = []
metrics_total = []

for data in dataset: 
    if data.csv_file is not None:
        position_error_total.extend(data.position_error[:,0])
        metrics = data.get_metrics()
        if metrics is not None:
            metrics_total.append(metrics)

#Get mean of localization error and variance to determine when the localization error is anomaly big
position_error_total = np.array(position_error_total)
mean = position_error_total.mean()
std = position_error_total.std()
umbral_position_error = mean + 3 * std
print( "umbral_position_error: " , umbral_position_error)

#Calculate navigation performance training the isolation tree
summary = pd.DataFrame(metrics_total)

num_cols = ['mean_pos_error', 'rmse_pos', 'max_pos_error', 'mae_yaw', 'executed_path_length', 'time_spent']
feat_df = summary[num_cols].fillna(-1)
if len(feat_df) > 0 and feat_df.shape[0] >= 3:
    iso = IsolationForest(random_state=42, contamination=0.1)
    try:
        iso.fit(feat_df)
    except Exception as e:
        print('IsolationForest failed:', e)


umbral_position_error:  0.877198388027938


In [12]:
#Get Anomalies

all_anomalies = []
all_anomalies_f_run = []
unique_anomalies = []
unique_anomalies_f_run = []


for data in dataset:
    data.clean_anomalies()
    data.detect_log_anomalies()
    #data.detect_time_anomalie(scenario_group) #ONCE THIS IS SOLVED WE CAN USE IT
    data.detect_localization_anomalie(umbral_position_error)
    data.detect_navigation_performance(iso)

    unique_a = data.get_unique_anomalies()
    unique_anomalies.extend(unique_a)
    all_anomalies.extend(data.anomalies)
    if data.failure_run:
        all_anomalies_f_run.extend(data.anomalies)
        unique_anomalies_f_run.extend(unique_a)

